In [ ]:
## In this tutorial, we will compute the thermodynamic properties for the b3lpy_opt state that we found in Tutorial 2, which includes the vibrational normal modes of a molecule 
## Also, we will discuss the DATA class, which is a very basic class that is used to store results

In [ ]:
import os
from Scope.Read_Write import load_binary

In [ ]:
## Path of the data folder. It should be "os.path.abspath('.')+'/Data"
data_folder = os.path.abspath('.')+'/Data/'
## Loads the System object from a binary file, provided in the tutorial folder
sys = load_binary(f"{data_folder}ABITEM.npy")

In [ ]:
## Then, we select the source 
found, source = sys.find_source("ref_hs_mol")

## And the state
found, final_state = source.find_state("b3lyp_opt")
print(final_state)

## Vibrational Normal Modes (VNMs)

In [ ]:
## Notice that the state is a minimum energy structure:
print(final_state.isminimum)

In [ ]:
## This is because the user provided the VNM. Here's the first one
print(final_state.VNMs[0])

In [ ]:
# By default, the eigenvectors of the VNMs are not stored... 
print(final_state.VNMs[0].has_mode)

In [ ]:
# ... but they can be parsed if needed. To do it:
freq_comp = final_state.computations[-1]                                       ## Select the computation
freq_comp.out_path = f"{data_folder}ABITEM_ref_hs_mol_freq_r1.log"             ## Modify is original path to re-read it

# And parse the frequencies
from Scope.Register_Data import reg_frequencies
worked = reg_frequencies(freq_comp, witheigen=True)
print(final_state.VNMs[0].has_mode)

## Thermodynamic Properties

In [ ]:
# Because the state contains the VNMs, we can compute the Thermodynamic Properties:
final_state.get_thermal_data()

In [ ]:
# The above function computes the vibrational enthalpy (Hvib) and entropy (Svib), as defined in the manuscript. 
# These, together with the electronic enthalpy (Helec) and entropy (Selec) are used to compute the Gibbs Free Energy (Gtot)
print(final_state.results["Helec"])
print(final_state.results["Selec"])


In [ ]:
# Notice that Helec and Selec are just a single instance of a so called DATA-class.
final_state.results["Helec"].type

In [ ]:
# The DATA class forces the results to be stored together with the units, and the function where they're computed.
print(final_state.results["Helec"].value)
print(final_state.results["Helec"].units)
print(final_state.results["Helec"].function)

In [ ]:
# In turn, Hvib, Svib and Gtot are a so-called COLLECTION class. This class gathers several data entries into a single instance 
print(final_state.results["Hvib"])
print(final_state.results["Svib"])
print(final_state.results["Gtot"])

In [ ]:
# To retrieve the values out of a COLLECTION class, you can:
print(final_state.results["Hvib"].get_values()[0:10]) # Notice that only the first 10 entries are printed


In [ ]:
# Again, units and the used function are specified:
print(final_state.results["Hvib"].units)
print(final_state.results["Hvib"].function)
# Also, what is the variable that changes within the collection:
print(final_state.results["Hvib"].variable)

In [ ]:
# To access an individual entry, you can search by the name of variable, and its value
final_state.results["Hvib"].find_value_with_property("temperature", 200)

### Subtracting Collections

In [ ]:
# Collections can be subtracted:
HS_state = final_state # we were operating with the HS molecule's data

## Lets select the LS as well, and compute its TD properties 
found, source   = sys.find_source("ref_ls_mol")
found, LS_state = source.find_state("b3lyp_opt")
LS_state.get_thermal_data()

print(HS_state)
print(LS_state)

In [ ]:
# Collections to substract
HS_Gtot = HS_state.results["Gtot"]
LS_Gtot = LS_state.results["Gtot"]

# New collection:
dGtot = HS_Gtot - LS_Gtot

# There is a simple function to plot Collections 
dGtot.view()

### Computation of T12 (SCO systems)

In [ ]:
from Scope.Spin_Crossover.SCO_Functions import get_T12

## For SCO systems, the transition temperature associated with the thermal spin transition LS->HS can be computed.

## The branch is the platform where the results will be stored
found, branch = sys.find_branch("Isolated")
## And the T12 is computed
worked, t12 = get_T12(branch, HS_state, LS_state, flexible=True)
print(worked)
print(t12)
# In this case, there is no T12, since the HS state is always more stable than the LS one. You can see it in the dGtot plot above

## Units Conversion

In [ ]:
## A very basic system to change units has been implemented at the DATA class level (not for collections).
print(final_state.results["Helec"])

## print_in_units() will preserve the original units:
print(final_state.results["Helec"].print_in_units("ev"))
print(final_state.results["Helec"])

## Convert to units function will overwrite the DATA units permanently:
print(final_state.results["Helec"].convert_to_units("cm"))
print(final_state.results["Helec"])